# LLM2-Lab — infraestrutura na A100

Selecione um runtime GPU A100. Este notebook verifica hardware, instala o projeto, testa, prepara dados sintéticos e demonstra treino BF16 e retomada. Não demonstra qualidade linguística ou ganho de desempenho. A disponibilidade de A100 depende do Colab.

No computador local: `python scripts/package_colab.py`. Envie `dist/llm2-lab-colab.zip` na célula abaixo. Depois substitua o corpus sintético pelos seus documentos UTF-8 separados por linhas vazias para iniciar pesquisa.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import zipfile

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
assert torch.cuda.is_available(), 'Selecione um runtime GPU no Colab.'
properties = torch.cuda.get_device_properties(0)
assert 'A100' in properties.name, f'Este protocolo pede A100; encontrada: {properties.name}'
assert properties.total_memory >= 39 * 1024**3, 'É necessária A100 com pelo menos 40 GB nominais.'
assert torch.cuda.is_bf16_supported(), 'BF16 indisponível.'
print(torch.__version__, torch.version.cuda, properties.name, properties.total_memory / 1024**3, 'GiB')


In [ ]:
from google.colab import files
ROOT = Path('/content/LLM2-Lab')
if not (ROOT / 'pyproject.toml').exists():
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    assert len(archives) == 1, 'Envie um único ZIP criado por scripts/package_colab.py.'
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.namelist():
            target = (ROOT / member).resolve()
            assert target.is_relative_to(ROOT.resolve()), 'Caminho inválido no ZIP.'
        archive.extractall(ROOT)
os.chdir(ROOT)
# Usa o Torch CUDA já presente se ele satisfizer o pyproject; não instala wheel CPU.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])

def run(*args):
    subprocess.check_call([sys.executable, *map(str, args)], cwd=ROOT)

run('-m', 'pytest', '-q')


Persistência: habilite `USE_DRIVE` para gravar dados, métricas e checkpoints no Google Drive. Sem Drive, baixe os artefatos antes de encerrar o runtime. Checkpoints são diretórios imutáveis; no Drive/FUSE, interrupções de I/O ainda podem exigir repetir a cópia. Os checksums detectam cópias incompletas.


In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE = Path('/content/drive/MyDrive/LLM2-Lab')
else:
    STORAGE = ROOT
STORAGE.mkdir(parents=True, exist_ok=True)
print('Artefatos em', STORAGE)


In [ ]:
RAW = STORAGE / 'data/raw/demo.txt'
DATA = STORAGE / 'data/processed-bpe'
if not RAW.exists():
    run('scripts/make_demo_corpus.py', '--output', RAW)
if not DATA.exists():
    run('-m', 'llm2', 'prepare', '--input', RAW, '--output', DATA,
        '--tokenizer', 'bpe', '--vocab-size', '8192', '--seed', '42')
from llm2.data import PreparedData
prepared = PreparedData(DATA)
print({key: value['tokens'] for key, value in prepared.manifest['splits'].items()})
print('Vocabulário efetivo:', prepared.tokenizer.vocab_size, '(modelo: 8192)')


Demonstração: 20 atualizações, com parada voluntária na etapa 10 e retomada. O horizonte do scheduler permanece em 5.000. Para um experimento real, crie um novo corpus, escolha um novo `RUN_NAME`, aumente `MAX_STEPS` e defina o horizonte antes da primeira atualização.

`COMPILE=True` ativa Inductor, cujo warmup entra na medição de tempo. `DETERMINISTIC=True` força SDPA math e algoritmos determinísticos; compare execuções só no mesmo ambiente.


In [ ]:
import yaml
RUN_NAME = 'transformer-34m-a100-demo'
MAX_STEPS = 20
FIRST_STAGE = 10
COMPILE = False
DETERMINISTIC = False
config = yaml.safe_load((ROOT / 'configs/transformer_34m_a100.yaml').read_text())
config['data']['path'] = str(DATA)
config['train'].update(output_dir=str(STORAGE / 'runs' / RUN_NAME),
                       max_steps=MAX_STEPS, eval_interval=10, checkpoint_interval=10,
                       log_interval=1, compile=COMPILE, deterministic=DETERMINISTIC)
CONFIG = ROOT / 'configs/colab_run.yaml'
CONFIG.write_text(yaml.safe_dump(config), encoding='utf-8')
run('-m', 'llm2', 'inspect', '--config', CONFIG)


In [ ]:
import json
RUN = Path(config['train']['output_dir'])
LATEST = RUN / 'checkpoints/latest.json'
if not LATEST.exists():
    run('-m', 'llm2', 'train', '--config', CONFIG, '--until-step', FIRST_STAGE)
checkpoint = RUN / 'checkpoints' / json.loads(LATEST.read_text())['checkpoint']
run('-m', 'llm2', 'train', '--config', CONFIG, '--resume', checkpoint)
checkpoint = RUN / 'checkpoints' / json.loads(LATEST.read_text())['checkpoint']
print('Checkpoint final:', checkpoint)


In [ ]:
run('-m', 'llm2', 'evaluate', '--checkpoint', checkpoint, '--data', DATA,
    '--device', 'cuda', '--precision', 'bf16')
run('-m', 'llm2', 'generate', '--checkpoint', checkpoint, '--data', DATA,
    '--device', 'cuda', '--precision', 'bf16', '--prompt', 'O laboratório',
    '--max-new-tokens', 40, '--seed', 42)
summary = json.loads((RUN / 'summary.json').read_text())
print(json.dumps(summary, indent=2))


Métricas: `metrics.jsonl`, `summary.json` e `tensorboard/` no diretório da execução. Para visualizar no Colab, execute a célula opcional abaixo. Tempos de GPU usam eventos CUDA por optimizer-step; não representam soma dos tempos individuais de kernels. Não compare o corpus sintético com benchmarks de linguagem.


In [ ]:
SHOW_TENSORBOARD = False
if SHOW_TENSORBOARD:
    ipython = get_ipython()
    ipython.run_line_magic('load_ext', 'tensorboard')
    ipython.run_line_magic('tensorboard', f'--logdir {RUN / "tensorboard"}')
